# Open Addressing<br/>Linear and Quadratic Probing


## TL/DR

Write code to measure frequency of collisions as a function of $\alpha$ in an openly addressable hash table where the `hash` of a string entry is the key for that entry.

_Hint 1:_ counting probes is a perfectly acceptable proxy for frequency of collisions.<br/>_Hint 2:_ you may want to read the rest of this notebook.

## Reading

* [Hashing](https://opendsa-server.cs.vt.edu/ODSA/Books/Everything/html/index.html#hashing) and probing techniques from the OpenDSA project.

## Collisions are inevitable

**Open addressing** is a collision resolution strategy.

A **collision** occurs when two distinct entries produce the same index after being passed through the hash function. For example, given the position function below:


In [ ]:
def position(entry: str, m: int) -> int:
    return hash(entry) % m

strings `Alice` and `Charlie` hash to the same slot of the underlying array when `m=4`. If we use function `position` above to store these strings in a list of size `m`, we'll have a collision.

Only one entry can occupy a given slot, of course, so we need a strategy to address collision.

One way to resolve the collision is **chaining.** It uses linked lists to hold multiple entries, at the same array (list) position, as shown below.

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/data_structures/chaining.png)

Chaining is demonstrated in [class `SimpleHashTable`](./live_coding.ipynb).


**Probing** is another way to resolve collisions. Using the example above, if `Alice` gets to position `[2]` first, and the position next to it is open, then `Charlie` could go there.

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/data_structures/probing.png)

In this case all entries are stored directly in the array itself — there are no external chains of linked lists. When a collision occurs (two entries hash to the same index), we search for the next available slot within the array by following a deterministic **probe sequence**. The general form is:

```python
probe(entry, i) = (hash(entry) + f(i)) % m
```

where `m` is the table capacity, `i` is the attempt number (0, 1, 2, …), and `f(i)` defines the probing strategy (e.g., `f(i) = i` for linear, `f(i) = i*i` for quadratic).


## Load factor

The load factor of a hash table is defined as

$$
\alpha = \frac{\text{elements of underlying array used}}{\text{length of underlying array}}
$$

In the purple example above, ${\color{purple}{\alpha = 1/4}}$ and in the orange example ${\color{orange}{\alpha = 2/4}}$. Obviously the load factor $\alpha$ can never exceed 1. At $\alpha=1$ every slot of the array is occupied and it must be resized. In practice, performance degrades well before that point, so implementations typically resize the underlying array as soon as $\alpha$ exceeds a threshold.


## General Probing Framework

Given a table of capacity `m` and a hash function `hash(entry)`, the probe sequence for a key is:

```python
probe(entry, i) = (hash(entry) + f(i)) % m  #  for i = 0, 1, 2, ...
```

where `f(i)` is the **probe function** and `i` is the attempt number (starting at 0). The two schemes differ only in their choice of `f(i)`.

| Scheme    | $f(i)$       |
| --------- | ------------ |
| Linear    | $f(i) = i$   |
| Quadratic | $f(i) = i^2$ |


## Linear Probing

On a collision at index `position`, we simply check the next slot, then the next, and so on, _wrapping_ around the array.

### Primary Clustering

Linear probing's main weakness is **primary clustering**. Once a contiguous block of occupied slots forms, any new entry that hashes into that block must traverse the entire cluster before finding an empty slot. Worse, doing so _extends_ the cluster, making future collisions even more expensive. As load factor increases, clusters merge and probe lengths grow rapidly.

### Properties

- Cache-friendly: probes visit consecutive memory locations.
- Simple to implement.
- Performance degrades sharply as the load factor $\alpha$ approaches 1.
- Average successful search: approximately $(1+1/(1-\alpha))/2$.
- Average unsuccessful search: approximately $(1 + (1/(1 − \alpha))^2)/2$.


## Quadratic Probing

On a collision, the probe jumps by 1, then 4, then 9, then 16, etc. The increasing step sizes help probes "escape" clusters more quickly.

### Secondary Clustering

Quadratic probing eliminates primary clustering but introduces **secondary clustering**: all keys that hash to the _same_ index follow the _same_ quadratic probe sequence. This is less severe than primary clustering but still causes extra work when many keys share a hash value.

### The Coverage Guarantee

Quadratic probing is **not guaranteed to visit every slot**. However, there is an important theorem:

> If $m$ is prime and the table is less than half full ($\alpha < 0.5$), then quadratic probing will always find an empty slot.

This is why implementations using quadratic probing typically choose a prime table size and resize when the load factor exceeds 0.5.

### Properties

- Reduces primary clustering.
- Slightly less cache-friendly than linear probing (probes are not contiguous).
- Requires care with table size (prime) and load factor (< 0.5 for the guarantee).
- Subject to secondary clustering.


## Side-by-Side Comparison

| Property             | Linear Probing       | Quadratic Probing         |
| -------------------- | -------------------- | ------------------------- |
| Probe function       | $f(i) = i$           | $f(i) = i^2$              |
| Primary clustering   | Yes (main weakness)  | No                        |
| Secondary clustering | No                   | Yes                       |
| Cache performance    | Excellent            | Good                      |
| Guaranteed insertion | Always (if not full) | Only if m is prime, α<0.5 |
| Typical max load     | ~0.7 practical limit | ~0.5 for the guarantee    |
| Implementation       | Simplest             | Slightly more complex     |


## Deletion: The Tombstone Problem

With open addressing, you cannot simply mark a slot as empty when deleting. Doing so would break the probe chain for keys that were inserted _past_ the deleted slot. The standard solution is **lazy deletion** using a **tombstone** (a special sentinel value):

- On delete: replace the slot with a tombstone marker.
- On search: treat tombstones as occupied (keep probing past them).
- On insert: treat tombstones as available (reuse the slot).

Over time, tombstones accumulate and degrade performance. A periodic rehash into a fresh table cleans them out.

## Key Takeaways

1. Both schemes store all entries directly in the array with no auxiliary data structures.
2. They differ in a single line of code: the probe function $f(i)$.
3. Linear probing is simple and cache-friendly but suffers from primary clustering.
4. Quadratic probing mitigates clustering but requires a prime table size and a lower load factor for its insertion guarantee.
5. Neither scheme handles deletion cleanly — tombstones are required.
6. In practice, keeping the load factor well below the theoretical limits (resize and rehash when needed) is essential for both schemes.


## Assignment 10

**Specifications and requirements.**  Each assignment must be compliant with the [Programmer's Pact](../housekeeping/ProgrammerPact_Python_2026.pdf).


- You may **not** import any modules **except** for those already included in the codebase of the assignment or listed below:
  - `from __future__ import annotations`
- No sets or dictionaries may be used.
- If your work requires additional methods to support the development of the methods the assignment asks for, you may write them.
- It's ok to deviate from the Pact and use protected (single underscore: `_`) instead of private (double underscores: `__`) members for your object classes. For example,
   ```python
   self._object_field = some_value  # Instead of self.__object_field = some_value
   ...
   def _internal_method(self):  # Instead of def __internal_method(self): 
   ```

## Measuring probes per insertion in open addressing

In this assignment, you will use the given `ProbingHashTable` class to run an experiment.

Your goal is to measure how many probes are needed, on average, for each insertion as the hash table becomes more full. You will do this for both:

* linear probing
* quadratic probing

Then you will organize the data so it can be plotted as:

* **x-axis:** load factor
* **y-axis:** average probes per insertion

The comparison should cover the **entire range of load factors**, from `0.0` up to `1.0`.

---

## What you are measuring

Each time you insert a value into the table, the `insert` method returns a list of the table positions that were probed.

So for one insertion:

* if the returned list has length `1`, then that insertion took `1` probe
* if the returned list has length `4`, then that insertion took `4` probes

Your experiment should measure the average number of probes required for insertions at different load factors.

---

## High-level idea

You will simulate many insertions into many fresh hash tables.

As the table fills up, you will record:

* the current load factor
* the number of probes used by the insertion

You should do this separately for:

* one set of trials using linear probing
* one set of trials using quadratic probing

Because one random trial can be noisy, you should repeat the experiment many times and average the results.

---

## Suggested step-by-step plan

### Step 1: Choose a table capacity

Pick a fixed table capacity, such as a reasonably sized prime number.

Why prime? It usually gives cleaner behavior for open addressing experiments.

---

### Step 2: Generate random strings

Create random strings to use as entries for insertion.

Each string should be different enough that you are not constantly reinserting the same value by accident.

---

### Step 3: Run one trial for one probing mode

For one trial:

1. Create a new empty `ProbingHashTable`
2. Repeatedly generate a random string and insert it
3. After each insertion attempt, determine:

   * how many probes were used
   * what the table’s load factor is now
4. Store that information

Do this once with mode `"linear"` and once with mode `"quadratic"`.

---

### Step 4: Decide how to associate an insertion with a load factor

For each successful insertion, record a pair like this:

* load factor after the insertion
* probes used by that insertion

Example idea:

* after the 1st successful insertion into a table of capacity 100, the load factor is `0.01`
* after the 2nd successful insertion, the load factor is `0.02`
* ...
* after the 100th successful insertion, the load factor is `1.00`

This gives you a natural set of x-values.

---

### Step 5: Repeat many trials

Run the same experiment many times for each probing mode.

For example, if the table capacity is `m`, then:

* Trial 1 gives probe counts for load factors `1/m`, `2/m`, `3/m`, ...
* Trial 2 gives another set
* and so on

Then average the probe counts for matching load factors.

So in the end, for each load factor, you will have:

* average probes for linear probing
* average probes for quadratic probing

---

### Step 6: Build a data table

Create a table with columns such as:

* load factor
* average probes, linear
* average probes, quadratic

This table is the main output of your simulation.

You can then import this data into a spreadsheet and create a graph.

---

## Important issue: failed insertions

This assignment asks for data over the **entire** load factor range from `0.0` to `1.0`.

That creates an important problem:

### Linear probing

Linear probing should eventually find an empty slot as long as one exists.

### Quadratic probing

Quadratic probing may fail to find an empty slot **even when the table is not full**.

That is not a bug in your simulation. It is part of the behavior you are studying.

---

## How to handle failed insertions

You need a consistent policy.

### What counts as a failed insertion?

An insertion attempt has failed if:

* the probe sequence does not find an empty slot
* therefore the table size does not increase

In other words, you attempted an insertion, but the table’s load factor did not move forward.

---

## Recommended strategy

When an insertion fails before the table is completely full:

1. **Do not treat that attempt as a successful insertion**
2. **Do not assign it a new load factor**
3. Generate a different random string and try again
4. Keep track of how many failed attempts occurred at the current load factor

This lets the simulation continue trying to reach larger load factors.

---

## Why this matters

For quadratic probing, near high load factors, you may discover that:

* some insertions succeed
* some insertions fail
* eventually it becomes extremely hard, or practically impossible, to place another key

That itself is useful experimental evidence.

---

## Two reasonable ways to report failures

Either of the following two approaches is acceptable, as long as you explain which one you use.

### Option A: Average probes over successful insertions only

For each load factor, average only the insertions that actually succeeded.

This is the simpler approach.

In that case, also keep a separate count of failed attempts so you can comment on them in your writeup.

### Option B: Record failure statistics separately

In addition to average probes for successful insertions, also record something like:

* number of failed insertion attempts
* failure rate at each load factor

This gives a fuller picture, especially for quadratic probing near the upper end.

---

## Practical tip for the full range from 0.0 to 1.0

Because quadratic probing may stall before the table reaches load factor `1.0`, you should not assume every single trial will produce data all the way to the end.

A good approach is:

* keep running many fresh trials
* average whatever successful data you obtain at each load factor
* note where quadratic probing begins to struggle or fail often

If you cannot reliably obtain data for the highest load factors with quadratic probing, say so explicitly and report the failure behavior as part of the result.

That is a valid outcome of the experiment.

---

## What to submit

File `week10.py` with your code. If you modified class `ProbingHashTable` include it in your code.

File `week10.*` with a picture of your data plotted. `*` could be PDF, PNG, or JPG.

File `week10.txt` with a brief discussion about the results you obtained from your simulation. For example:

* How does average probes per insertion change as load factor increases?
* How do linear and quadratic probing compare at low load factors?
* How do they compare at high load factors?
* Does quadratic probing begin to fail before the table is full?
* If so, around what region of the load factor does this become noticeable?

---

## Plotting suggestion

You do not need to produce the graph in Python. Export your data to a spreadsheet and create a line chart there. Your chart should compare linear and quadratic probing on the same axes.

---

## Small implementation hints

A few practical suggestions for better results:

* Use many trials, not just one
* Use fresh empty tables for each trial
* Use random strings as inserted values
* Keep linear and quadratic results separate
* Be careful not to mix up:

  * load factor before insertion
  * load factor after insertion
* Decide in advance how you will handle failed insertions, and be consistent



---

# Class `ProbingHashTable`

This implementation does not use the typical `key, value` pair. Instead it stores string entries based on their hash code. The `index` of a string `entry` in the underlying array is computed as 

```python
index =  abs(hash(entry)) % self._capacity
```

If that position, however, is occupied we begin probing at index positions
```python
(index + f(i)) % self._capacity
```
where $f(i)=i$ for linear probing or $f(i)=i^2$ for quadratic, with $0 < i < \texttt{self.\_capacity}$.

In [ ]:
from __future__ import annotations


class ProbingHashTable:
    """Hash table demonstrating linear vs. quadratic probing."""

    # Consstants for default values and error messages
    _DEFAULT_CAPACITY = 11
    _PROBE_MODES = ("linear", "quadratic")
    _DEFAULT_PROBING_MODE = _PROBE_MODES[0]
    _ERROR_MODE = f"mode must be one of {', '.join(_PROBE_MODES)}"
    _ERROR_FULL = "Hash table is full"

    def __init__(
        self, capacity: int = _DEFAULT_CAPACITY, mode: str = _DEFAULT_PROBING_MODE
    ):
        """Initialize the hash table with the given capacity and probing mode."""
        if mode not in self._PROBE_MODES:
            # If the mode is not valid, 
            # use default mode.
            mode = self._DEFAULT_PROBING_MODE
        self._capacity = capacity
        self._mode = mode
        # Each slot is a string or None
        self._table: list[str | None] = [None] * capacity
        self._size = 0

    def _hash(self, key: int) -> int:
        """Convert a key to an index in the underlying table."""
        return key % self._capacity

    def _probe(self, position: int, attempt: int) -> int:
        ""
        if self._mode == self._DEFAULT_PROBING_MODE:  # linear
            return (position + attempt) % self._capacity
        else:  # quadratic
            return (position + attempt * attempt) % self._capacity
        
    def load_factor(self) -> float:
        """Return the current load factor of the hash table."""
        return self._size / self._capacity

    def insert(self, value: str) -> list[int]:
        """Insert key-value pair. Returns list of indices probed."""
        # Initialize list to track probe indices
        probes = []
        # Check if the hash table has space for a new entry.
        # If not, we will not attempt to insert and will return
        # an empty list of probes.
        if self._size < self._capacity:
            # Generate a non-negative integer key from the value
            key: int = abs(hash(value))
            # Convert the key to an index in the underlying table
            index = self._hash(key)
            # Attempt to insert the key-value pair, probing for an empty
            # slot.
            i: int = 0
            insertion_successful: bool = False
            # Loop to attempt to find an empty slot. The loop ends as soon as
            # we have probed the entire table or successfully inserted the
            # value.
            while i < self._capacity and not insertion_successful:
                # Location in the underlying table to check
                probe_index = self._probe(index, i)
                # Record the location we are probing
                probes.append(probe_index)
                # Obtain the contents at the location we are probing
                slot = self._table[probe_index]
                # Determine if we can insert the value at the location we are probing.
                # If insertion is successful, we will exit the loop. If not, we will 
                # try the next probe index in the next iteration of the loop.
                insertion_successful = slot is None or slot == value
                if insertion_successful:
                    # The slot is empty or contains the same value already, 
                    # so we can insert the value pair here (or update the existing 
                    # value if it is the same).
                    self._table[probe_index] = value
                    if slot is None:
                     # Update the size of the hash table if we added
                     # this value for the first time, ie, we have not
                     # overwriten it.
                        self._size += 1
                    

                # If the slot is not empty, we will try the next probe index
                # in the next iteration of the loop.
                i += 1
        return probes

    def contains(self, value: str) -> bool:
        """Check if value is in the hash table."""
        # Generate a non-negative integer key from the value
        key = abs(hash(value))
        # Convert the key to an index in the underlying table
        index = self._hash(key)
        # Loop to probe for the value. The loop ends as soon as we have probed the
        # entire table or found the value.
        found: bool = False
        i: int = 0
        # Loop ends when we have probed the entire table or found the value
        while i < self._capacity and not found:
            probe_index = self._probe(index, i)
            slot = self._table[probe_index]
            found = slot == value
            i += 1
        return found

    # Formating constants for display
    _FMT_HEADER = f"{'Idx':<5}{'Content'}"
    _FMT_EMPTY = "---"
    _FMT_SLOT = "{idx:5} -> {content}"
    _FMT_HORIZONTAL = "-" * 20

    def display(self) -> str:
        """Return a string representation of the hash table."""
        lines = [self._FMT_HEADER]
        lines.append(self._FMT_HORIZONTAL)
        for i in range(self._capacity):
            slot = self._table[i]
            content = self._FMT_SLOT.format(idx=i, content=slot) if slot else self._FMT_EMPTY
            lines.append(content)
        return "\n".join(lines)


# ── Demo ──────────────────────────────────────────────────────

import random
import string
capacity = 11

def random_string(n: int) -> str:
    return "".join(random.choices(string.ascii_letters, k=n))   
    
def demo(trials:int):

    for mode in ("linear", "quadratic"):
        ht = ProbingHashTable(capacity=capacity, mode=mode)
        print(f"\n{'=' * 40}")
        print(f"  {mode.upper()} PROBING (capacity={capacity})")
        print(f"{'=' * 40}")

        for _ in range(trials):
            value = random_string(5)
            probes = ht.insert(value)
            status = "collision!" if len(probes) > 1 else "direct"
            print(f"  insert({value}): hash={abs(hash(value)) % capacity:<3} probes={probes}  [{status}]")

        print(f"\n{ht.display()}")
        print(
            f"\nLoad factor: {ht._size}/{ht._capacity} = {ht._size / ht._capacity:.2f}"
        )


if __name__ == "__main__":
    demo(10)